# Experiment 6: Comparing isotropic versus standard-tanh

This notebook performs a paired fixed-architecture comparison between standard coordinatewise `tanh` MLPs and `IsotropicTanhMLP` networks.

Across datasets: MNIST, Fashion-MNIST, CIFAR-10, CIFAR-100, Caltech256 resized to 64x64, EMNIST A--J; four repeats and 50 epochs per permutation.


In [ ]:
# -------------------------
# Imports and paths
# -------------------------

import sys
import pickle as pkl
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import time
import math
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset

import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output

from torchvision import datasets, transforms
from torchvision.datasets.utils import download_and_extract_archive

# Ensure the uploaded project modules are importable when this notebook is run from its own directory.
PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from Dependencies import *

RESULTS_DIR = Path("Experiment6_results")
RUNS_DIR = RESULTS_DIR / "runs"
TABLES_DIR = RESULTS_DIR / "tables"
PLOTS_DIR = RESULTS_DIR / "plots"
SUMMARY_DIR = RESULTS_DIR / "summaries"

for directory in [RESULTS_DIR, RUNS_DIR, TABLES_DIR, PLOTS_DIR, SUMMARY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# -------------------------
# Hyperparameters
# -------------------------

DATA_ROOT = Path("./data")
NORMALISATION_DIR = Path(".")
NORMALISATION = True
CALTECH256_IMAGE_SIZE = 64
CALTECH256_TRAIN_FRACTION = 0.8
CALTECH256_SPLIT_SEED = 20260504

DATASET_ORDER = ["MNIST", "FMNIST", "CIFAR10", "CIFAR100", "CALTECH256", "EMNIST_AJ"]
MODEL_TYPES = ["standard_tanh", "isotropic_tanh"]

# Hidden-layer architecture templates. Full architecture is [input_width, *hidden_widths, num_classes].
HIDDEN_ARCHITECTURES = [
    [500, 500],
    [500, 500, 500],
    [500, 500, 500, 500],
    [500, 500, 500, 500, 500],
    [1000, 1000],
    [1000, 1000, 1000],
    [1000, 1000, 1000, 1000],
    [1000, 1000, 1000, 1000, 1000],
]

REPEATS = 4
TOTAL_EPOCHS = 50
BATCH_SIZE = 48
LEARNING_RATE = 1e-3
ADAMW_WEIGHT_DECAY = 1e-3
WEIGHT_INIT = "orthogonal"

# Keep false by default to avoid storing very large model checkpoints for every run.
# The per-run result/history pickle is always saved and is used for resuming/skipping.
SAVE_MODEL_STATES = False

# Strict fair-comparison settings for the isotropic network.
ISOTROPIC_INTRINSIC_LENGTH_APPROACH = "None"
ISOTROPIC_LINEAR_CORRECTION_APPROACH = "None"

assert ISOTROPIC_INTRINSIC_LENGTH_APPROACH.upper() == "NONE"
assert ISOTROPIC_LINEAR_CORRECTION_APPROACH.upper() == "NONE"

SEED_GENERATOR_SEED = 20260501
seed_rng = np.random.default_rng(SEED_GENERATOR_SEED)
REPEAT_ORTHOGONAL_INIT_SEEDS = seed_rng.choice(
    np.arange(1_000_000, 999_999_999, dtype=np.int64),
    size=REPEATS,
    replace=False,
).astype(int).tolist()
REPEAT_DATALOADER_SEEDS = seed_rng.choice(
    np.arange(1_000_000, 999_999_999, dtype=np.int64),
    size=REPEATS,
    replace=False,
).astype(int).tolist()

DEVICE = try_gpu(output=True)
LOSS = nn.CrossEntropyLoss()

print(f"Device: {DEVICE}")
print(f"Datasets: {DATASET_ORDER}")
print(f"Hidden architectures: {HIDDEN_ARCHITECTURES}")
print(f"Repeats: {REPEATS}")
print(f"Orthogonal initialisation seeds per repeat: {REPEAT_ORTHOGONAL_INIT_SEEDS}")
print(f"Dataloader seeds per repeat: {REPEAT_DATALOADER_SEEDS}")
print(f"Epochs per run: {TOTAL_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"AdamW weight decay: {ADAMW_WEIGHT_DECAY}")
print(f"Isotropic intrinsic length approach: {ISOTROPIC_INTRINSIC_LENGTH_APPROACH}")
print(f"Isotropic linear correction approach: {ISOTROPIC_LINEAR_CORRECTION_APPROACH}")


In [ ]:
# -------------------------
# Dataset and normalisation utilities
# -------------------------

class RequiredPerPixelNormalize:
    """Per-pixel normaliser using existing .pkl files only; never creates them."""

    def __init__(self, path):
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(
                f"Required normalisation file not found: {path}. "
                "This notebook intentionally does not create normalisation .pkl files."
            )
        normaliser_dictionary = pkl.load(open(path, "rb"))
        self.mean = normaliser_dictionary["mean"].to(torch.float32)
        self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

    def __call__(self, tensor):
        return (tensor - self.mean) * self.inv_std


class FilteredLabelDataset(Dataset):
    """Filter a labelled dataset and remap labels, used for EMNIST A--J."""

    def __init__(self, dataset, allowed_labels, label_remap):
        self.dataset = dataset
        self.allowed_labels = set(int(label) for label in allowed_labels)
        self.label_remap = {int(k): int(v) for k, v in label_remap.items()}
        self.indices = []
        for idx in range(len(dataset)):
            _, label = dataset[idx]
            if int(label) in self.allowed_labels:
                self.indices.append(idx)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.dataset[self.indices[idx]]
        return image, self.label_remap[int(label)]


NORMALISATION_FILES = {
    "MNIST": NORMALISATION_DIR / "MNIST_normalisations.pkl",
    "FMNIST": NORMALISATION_DIR / "FMNIST_normalisations.pkl",
    "CIFAR10": NORMALISATION_DIR / "CIFAR_normalisations.pkl",
    "CIFAR100": NORMALISATION_DIR / "CIFAR_normalisations.pkl",
    "CALTECH256": NORMALISATION_DIR / f"CALTECH256_{CALTECH256_IMAGE_SIZE}x{CALTECH256_IMAGE_SIZE}_normalisations.pkl",
    "EMNIST_AJ": NORMALISATION_DIR / "EMNIST_AJ_normalisations.pkl",
}

DATASET_SPECS = {
    "MNIST": {"input_width": 28 * 28, "num_classes": 10},
    "FMNIST": {"input_width": 28 * 28, "num_classes": 10},
    "CIFAR10": {"input_width": 3 * 32 * 32, "num_classes": 10},
    "CIFAR100": {"input_width": 3 * 32 * 32, "num_classes": 100},
    "CALTECH256": {"input_width": 3 * CALTECH256_IMAGE_SIZE * CALTECH256_IMAGE_SIZE, "num_classes": 257},
    "EMNIST_AJ": {"input_width": 28 * 28, "num_classes": 10},
}


CALTECH256_ARCHIVE_MD5 = "67b4f42ca05d46448c6bb8ecd2220f6d"
CALTECH256_DOWNLOAD_URLS = [
    # Stable CaltechDATA record for Caltech-256. The torchvision Google Drive URL may return 404.
    "https://data.caltech.edu/records/nyy15-4j048/files/256_ObjectCategories.tar?download=1",
    # Legacy upstream URL retained as a fallback for older environments/mirrors.
    "http://www.vision.caltech.edu/Image_Datasets/Caltech256/256_ObjectCategories.tar",
]


def ensure_caltech256_files_available():
    """Ensure Caltech256 files are present without relying on torchvision's broken Google Drive URL."""
    caltech_root = DATA_ROOT / "caltech256"
    categories_dir = caltech_root / "256_ObjectCategories"
    if categories_dir.exists():
        return

    caltech_root.mkdir(parents=True, exist_ok=True)
    download_errors = []
    for url in CALTECH256_DOWNLOAD_URLS:
        try:
            download_and_extract_archive(
                url,
                download_root=str(caltech_root),
                filename="256_ObjectCategories.tar",
                md5=CALTECH256_ARCHIVE_MD5,
            )
            if categories_dir.exists():
                return
        except Exception as err:
            download_errors.append(f"{url}: {err}")

    raise RuntimeError(
        "Could not download/extract Caltech256 from any configured URL. "
        "You can also manually place 256_ObjectCategories under ./data/caltech256/.\n"
        + "\n".join(download_errors)
    )


def make_caltech256_base_transform():
    return transforms.Compose([
        transforms.Lambda(lambda image: image.convert("RGB")),
        transforms.Resize((CALTECH256_IMAGE_SIZE, CALTECH256_IMAGE_SIZE)),
        transforms.ToTensor(),
    ])


def ensure_caltech256_normalisation_file():
    """Create the Caltech256 64x64 per-pixel normalisation file if it is absent."""
    normalisation_path = NORMALISATION_FILES["CALTECH256"]
    if (not NORMALISATION) or normalisation_path.exists() or "CALTECH256" not in DATASET_ORDER:
        return

    ensure_caltech256_files_available()
    stat_dataset = datasets.Caltech256(
        root=str(DATA_ROOT),
        download=False,
        transform=make_caltech256_base_transform(),
    )
    stat_loader = DataLoader(stat_dataset, batch_size=256, shuffle=False, num_workers=0)

    pixel_sum = None
    pixel_square_sum = None
    image_count = 0

    for batch_images, _ in stat_loader:
        batch_images = batch_images.to(torch.float32)
        if pixel_sum is None:
            pixel_sum = torch.zeros_like(batch_images[0])
            pixel_square_sum = torch.zeros_like(batch_images[0])
        pixel_sum += batch_images.sum(dim=0)
        pixel_square_sum += (batch_images ** 2).sum(dim=0)
        image_count += int(batch_images.shape[0])

    caltech256_mean = pixel_sum / image_count
    caltech256_variance = torch.clamp(pixel_square_sum / image_count - caltech256_mean ** 2, min=0.0)
    caltech256_std = torch.sqrt(caltech256_variance)
    caltech256_inv_std = torch.where(
        caltech256_std > 1e-8,
        1.0 / caltech256_std,
        torch.ones_like(caltech256_std),
    )

    pkl.dump(
        {"mean": caltech256_mean.to(torch.float32), "inverse stddev": caltech256_inv_std.to(torch.float32)},
        open(normalisation_path, "wb"),
    )


def make_transform(dataset_name):
    transform_steps = []
    if dataset_name == "CALTECH256":
        transform_steps.extend([
            transforms.Lambda(lambda image: image.convert("RGB")),
            transforms.Resize((CALTECH256_IMAGE_SIZE, CALTECH256_IMAGE_SIZE)),
        ])
    transform_steps.append(transforms.ToTensor())
    if NORMALISATION:
        transform_steps.append(RequiredPerPixelNormalize(NORMALISATION_FILES[dataset_name]))
    return transforms.Compose(transform_steps)


def make_caltech256_split_indices(dataset_length):
    rng = np.random.default_rng(CALTECH256_SPLIT_SEED)
    indices = rng.permutation(dataset_length)
    train_count = int(round(CALTECH256_TRAIN_FRACTION * dataset_length))
    return indices[:train_count].tolist(), indices[train_count:].tolist()


def load_all_datasets():
    ensure_caltech256_normalisation_file()
    transforms_by_dataset = {name: make_transform(name) for name in DATASET_ORDER}

    dataset_dict = {}
    dataset_dict["MNIST"] = {
        "train": datasets.MNIST(root=str(DATA_ROOT), train=True, download=True, transform=transforms_by_dataset["MNIST"]),
        "test": datasets.MNIST(root=str(DATA_ROOT), train=False, download=True, transform=transforms_by_dataset["MNIST"]),
    }
    dataset_dict["FMNIST"] = {
        "train": datasets.FashionMNIST(root=str(DATA_ROOT), train=True, download=True, transform=transforms_by_dataset["FMNIST"]),
        "test": datasets.FashionMNIST(root=str(DATA_ROOT), train=False, download=True, transform=transforms_by_dataset["FMNIST"]),
    }
    dataset_dict["CIFAR10"] = {
        "train": datasets.CIFAR10(root=str(DATA_ROOT), train=True, download=True, transform=transforms_by_dataset["CIFAR10"]),
        "test": datasets.CIFAR10(root=str(DATA_ROOT), train=False, download=True, transform=transforms_by_dataset["CIFAR10"]),
    }
    cifar100_urls = [
        datasets.CIFAR100.url,
        "https://huggingface.co/datasets/nakroy/cifar100-python/resolve/main/cifar-100-python.tar.gz",
    ]
    cifar100_errors = []
    for cifar100_url in cifar100_urls:
        datasets.CIFAR100.url = cifar100_url
        try:
            dataset_dict["CIFAR100"] = {
                "train": datasets.CIFAR100(root=str(DATA_ROOT), train=True, download=True, transform=transforms_by_dataset["CIFAR100"]),
                "test": datasets.CIFAR100(root=str(DATA_ROOT), train=False, download=True, transform=transforms_by_dataset["CIFAR100"]),
            }
            break
        except Exception as err:
            cifar100_errors.append(f"{cifar100_url}: {err}")
    else:
        raise RuntimeError("Could not download/load CIFAR100 from any configured URL:\n" + "\n".join(cifar100_errors))

    ensure_caltech256_files_available()
    caltech256_full = datasets.Caltech256(
        root=str(DATA_ROOT),
        download=False,
        transform=transforms_by_dataset["CALTECH256"],
    )
    caltech256_train_indices, caltech256_test_indices = make_caltech256_split_indices(len(caltech256_full))
    dataset_dict["CALTECH256"] = {
        "train": Subset(caltech256_full, caltech256_train_indices),
        "test": Subset(caltech256_full, caltech256_test_indices),
    }

    emnist_train_raw = datasets.EMNIST(root=str(DATA_ROOT), split="letters", train=True, download=True, transform=transforms_by_dataset["EMNIST_AJ"])
    emnist_test_raw = datasets.EMNIST(root=str(DATA_ROOT), split="letters", train=False, download=True, transform=transforms_by_dataset["EMNIST_AJ"])
    aj_remap = {label: label - 1 for label in range(1, 11)}
    dataset_dict["EMNIST_AJ"] = {
        "train": FilteredLabelDataset(emnist_train_raw, allowed_labels=range(1, 11), label_remap=aj_remap),
        "test": FilteredLabelDataset(emnist_test_raw, allowed_labels=range(1, 11), label_remap=aj_remap),
    }
    return dataset_dict


ALL_DATASETS = load_all_datasets()
for name in DATASET_ORDER:
    print(f"{name}: train={len(ALL_DATASETS[name]['train'])}, test={len(ALL_DATASETS[name]['test'])}")


In [ ]:

class StandardTanhMLP(nn.Module):
    """Coordinatewise tanh MLP with parameter layout mirroring IsotropicTanhMLP."""

    def __init__(self, layers, flatten=True, device=None, dtype=None):
        super().__init__()
        factory_kwargs = {"device": device, "dtype": dtype}
        self.flatten = bool(flatten)
        self.weight_parameters = nn.ParameterList([
            nn.Parameter(torch.empty((out_features, in_features), **factory_kwargs))
            for in_features, out_features in zip(layers[:-1], layers[1:])
        ])
        self.bias_parameters = nn.ParameterList([
            nn.Parameter(torch.empty(out_features, **factory_kwargs))
            for out_features in layers[1:]
        ])

    @property
    def architecture(self):
        return [self.weight_parameters[0].shape[1]] + [bias.shape[0] for bias in self.bias_parameters]

    def simple_initialiser(self, weight_init="orthogonal"):
        with torch.no_grad():
            for weight in self.weight_parameters:
                if weight_init == "orthogonal":
                    nn.init.orthogonal_(weight)
                elif weight_init == "xavier_normal":
                    nn.init.xavier_normal_(weight)
                elif weight_init == "xavier_uniform":
                    nn.init.xavier_uniform_(weight)
                else:
                    raise ValueError(f"Unknown weight_init={weight_init}")
            for bias in self.bias_parameters:
                bias.zero_()

    def forward(self, x):
        if self.flatten:
            x = x.flatten(start_dim=1)
        temporary = x
        for weight, bias in zip(self.weight_parameters[:-1], self.bias_parameters[:-1]):
            temporary = torch.tanh(torch.einsum("ij,bj->bi", weight, temporary) + bias[None, :])
        output = torch.einsum("ij,bj->bi", self.weight_parameters[-1], temporary) + self.bias_parameters[-1][None, :]
        return output


def full_architecture(dataset_name, hidden_architecture):
    spec = DATASET_SPECS[dataset_name]
    return [spec["input_width"], *list(hidden_architecture), spec["num_classes"]]


def make_initial_affine_state(layers, seed):
    """Create one deterministic orthogonal affine state on CPU for later copying to either model type."""
    previous_rng_state = torch.random.get_rng_state()
    torch.manual_seed(int(seed))
    template = StandardTanhMLP(layers=layers, flatten=True, device="cpu")
    template.simple_initialiser(weight_init=WEIGHT_INIT)
    weights = [parameter.detach().clone() for parameter in template.weight_parameters]
    biases = [parameter.detach().clone() for parameter in template.bias_parameters]
    torch.random.set_rng_state(previous_rng_state)
    return weights, biases


def copy_affine_state(network, weights, biases):
    with torch.no_grad():
        for target, source in zip(network.weight_parameters, weights):
            target.copy_(source.to(device=target.device, dtype=target.dtype))
        for target, source in zip(network.bias_parameters, biases):
            target.copy_(source.to(device=target.device, dtype=target.dtype))


def assert_fair_isotropic_network(network):
    assert isinstance(network, IsotropicTanhMLP)
    assert network.intrinsic_length_approach.upper() == "NONE"
    assert network.linear_correction_approach.upper() == "NONE"
    assert not hasattr(network, "psi_parameters"), "psi_parameters should not exist for linear_correction_approach='None'."
    for activation in network.activation:
        assert activation.trainable is False
        assert torch.isclose(activation.o.detach().cpu(), torch.tensor(0.0)).item()


def build_model(model_type, layers, affine_seed, device):
    weights, biases = make_initial_affine_state(layers=layers, seed=affine_seed)
    if model_type == "standard_tanh":
        network = StandardTanhMLP(layers=layers, flatten=True, device=device)
    elif model_type == "isotropic_tanh":
        network = IsotropicTanhMLP(
            layers=layers,
            flatten=True,
            intrinsic_length_approach=ISOTROPIC_INTRINSIC_LENGTH_APPROACH,
            linear_correction_approach=ISOTROPIC_LINEAR_CORRECTION_APPROACH,
            device=device,
        )
        assert_fair_isotropic_network(network)
    else:
        raise ValueError(f"Unknown model_type={model_type}")
    copy_affine_state(network, weights, biases)
    return network


In [ ]:
# -------------------------
# Training, evaluation, cache, and progress-table utilities
# -------------------------

def make_dataloaders(dataset_name, repeat_idx):
    generator = torch.Generator()
    generator.manual_seed(int(REPEAT_DATALOADER_SEEDS[repeat_idx]))
    train_loader = DataLoader(
        ALL_DATASETS[dataset_name]["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=0,
        generator=generator,
    )
    test_loader = DataLoader(
        ALL_DATASETS[dataset_name]["test"],
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )
    return train_loader, test_loader


def evaluate_classifier(network, data_loader, device, loss_fn):
    network.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    with torch.no_grad():
        for batch_data, batch_labels in data_loader:
            batch_data = batch_data.to(device)
            batch_labels = batch_labels.to(device)
            logits = network(batch_data)
            loss_value = loss_fn(logits, batch_labels)
            batch_size = int(batch_labels.shape[0])
            total_loss += float(loss_value.detach().cpu()) * batch_size
            total_correct += int((torch.argmax(logits, dim=1) == batch_labels).sum().detach().cpu())
            total_count += batch_size
    return total_loss / total_count, 100.0 * total_correct / total_count


def train_one_epoch(network, data_loader, device, optimiser, loss_fn):
    network.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    for batch_data, batch_labels in data_loader:
        batch_data = batch_data.to(device)
        batch_labels = batch_labels.to(device)
        optimiser.zero_grad(set_to_none=True)
        logits = network(batch_data)
        error = loss_fn(logits, batch_labels)
        error.backward()
        optimiser.step()

        batch_size = int(batch_labels.shape[0])
        total_loss += float(error.detach().cpu()) * batch_size
        total_correct += int((torch.argmax(logits, dim=1) == batch_labels).sum().detach().cpu())
        total_count += batch_size
    return total_loss / total_count, 100.0 * total_correct / total_count


# -------------------------
# Robust architecture-aware cache utilities
# -------------------------

def architecture_label_from_hidden(hidden_architecture):
    return "-".join(str(int(width)) for width in hidden_architecture)


def architecture_tuple(hidden_architecture):
    return tuple(int(width) for width in hidden_architecture)


def current_architecture_index(hidden_architecture):
    target = architecture_tuple(hidden_architecture)
    for idx, candidate in enumerate(HIDDEN_ARCHITECTURES):
        if architecture_tuple(candidate) == target:
            return idx
    return None


def cache_key(dataset_name, hidden_architecture, model_type, repeat_idx):
    return (
        str(dataset_name),
        architecture_tuple(hidden_architecture),
        str(model_type),
        int(repeat_idx),
    )


def run_result_path(dataset_name, architecture_idx, hidden_architecture, model_type, repeat_idx):
    architecture_label = architecture_label_from_hidden(hidden_architecture)
    return RUNS_DIR / (
        f"repeat_{repeat_idx:02d}__{dataset_name}_CT256"
        f"__arch_{architecture_idx:02d}_{architecture_label}"
        f"__{model_type}.pkl"
    )


def model_state_path(dataset_name, architecture_idx, hidden_architecture, model_type, repeat_idx):
    architecture_label = architecture_label_from_hidden(hidden_architecture)
    return RUNS_DIR / (
        f"repeat_{repeat_idx:02d}__{dataset_name}_CT256"
        f"__arch_{architecture_idx:02d}_{architecture_label}"
        f"__{model_type}__state.pt"
    )


def legacy_run_result_path(dataset_name, architecture_idx, model_type, repeat_idx):
    return RUNS_DIR / f"repeat_{repeat_idx:02d}__{dataset_name}__arch_{architecture_idx:02d}__{model_type}.pkl"


def legacy_model_state_path(dataset_name, architecture_idx, model_type, repeat_idx):
    return RUNS_DIR / f"repeat_{repeat_idx:02d}__{dataset_name}__arch_{architecture_idx:02d}__{model_type}__state.pt"


def record_is_usable(record):
    required_keys = [
        "dataset_name",
        "hidden_architecture",
        "model_type",
        "repeat_idx",
        "final_test_accuracy",
        "test_accuracy_history",
    ]
    return all(key in record for key in required_keys)


def record_belongs_to_current_experiment(record):
    if not record_is_usable(record):
        return False

    if record["dataset_name"] not in DATASET_ORDER:
        return False

    if record["model_type"] not in MODEL_TYPES:
        return False

    if not (0 <= int(record["repeat_idx"]) < REPEATS):
        return False

    if current_architecture_index(record["hidden_architecture"]) is None:
        return False

    return True


def normalise_cached_record(record):
    """
    Normalise a cached record to the current HIDDEN_ARCHITECTURES ordering.

    This is the important fix.

    Older cache files may have a stale architecture_idx because that index depended on
    the position of the architecture in HIDDEN_ARCHITECTURES at the time the file was
    written. Here we recover the correct current architecture_idx from the actual
    hidden_architecture stored inside the record.
    """
    record = dict(record)

    hidden_architecture = list(record["hidden_architecture"])
    architecture_idx = current_architecture_index(hidden_architecture)

    if architecture_idx is None:
        raise ValueError(f"Cached architecture is not in current HIDDEN_ARCHITECTURES: {hidden_architecture}")

    dataset_name = record["dataset_name"]
    model_type = record["model_type"]
    repeat_idx = int(record["repeat_idx"])
    architecture_label = architecture_label_from_hidden(hidden_architecture)

    record["architecture_idx"] = int(architecture_idx)
    record["architecture_label"] = architecture_label
    record["hidden_architecture"] = hidden_architecture
    record["architecture"] = full_architecture(dataset_name, hidden_architecture)

    canonical_result_path = run_result_path(
        dataset_name=dataset_name,
        architecture_idx=architecture_idx,
        hidden_architecture=hidden_architecture,
        model_type=model_type,
        repeat_idx=repeat_idx,
    )

    canonical_state_path = model_state_path(
        dataset_name=dataset_name,
        architecture_idx=architecture_idx,
        hidden_architecture=hidden_architecture,
        model_type=model_type,
        repeat_idx=repeat_idx,
    )

    record["result_path"] = str(canonical_result_path)
    if record.get("model_state_path") is not None:
        record["model_state_path"] = str(canonical_state_path)

    return record


def record_matches_requested(record, dataset_name, hidden_architecture, model_type, repeat_idx):
    if not record_is_usable(record):
        return False

    return cache_key(
        record["dataset_name"],
        record["hidden_architecture"],
        record["model_type"],
        record["repeat_idx"],
    ) == cache_key(
        dataset_name,
        hidden_architecture,
        model_type,
        repeat_idx,
    )


def find_matching_cached_record(dataset_name, hidden_architecture, model_type, repeat_idx):
    matching_records = []

    for path in sorted(RUNS_DIR.glob("repeat_*__*.pkl")):
        try:
            with open(path, "rb") as handle:
                candidate = pkl.load(handle)

            if not record_belongs_to_current_experiment(candidate):
                continue

            candidate = normalise_cached_record(candidate)

            if record_matches_requested(
                candidate,
                dataset_name=dataset_name,
                hidden_architecture=hidden_architecture,
                model_type=model_type,
                repeat_idx=repeat_idx,
            ):
                matching_records.append((path, candidate))

        except Exception as exc:
            print(f"Warning: could not inspect cached file {path}: {exc}")

    if len(matching_records) == 0:
        return None

    if len(matching_records) > 1:
        print(
            "Warning: multiple matching cached records found for "
            f"repeat={repeat_idx}, dataset={dataset_name}, "
            f"architecture={architecture_label_from_hidden(hidden_architecture)}, "
            f"model={model_type}. Using the first after sorting."
        )
        for path, _ in matching_records:
            print(f"  matching cache file: {path}")

    original_path, record = matching_records[0]
    return record


def load_existing_records():
    records_by_key = {}

    for path in sorted(RUNS_DIR.glob("repeat_*__*.pkl")):
        try:
            with open(path, "rb") as handle:
                record = pkl.load(handle)

            if not record_belongs_to_current_experiment(record):
                print(f"Skipping stale or non-current cached record: {path}")
                continue

            record = normalise_cached_record(record)

            key = cache_key(
                record["dataset_name"],
                record["hidden_architecture"],
                record["model_type"],
                record["repeat_idx"],
            )

            if key in records_by_key:
                print(f"Warning: duplicate cached record for {key}; keeping latest encountered file: {path}")

            records_by_key[key] = record

        except Exception as exc:
            print(f"Warning: could not load {path}: {exc}")

    return list(records_by_key.values())


def records_to_dataframe(records):
    if len(records) == 0:
        return pd.DataFrame()

    rows = []
    for record in records:
        record = normalise_cached_record(record)

        rows.append({
            "dataset_name": record["dataset_name"],
            "architecture_idx": record["architecture_idx"],
            "architecture_label": record["architecture_label"],
            "architecture": str(record["architecture"]),
            "hidden_architecture": str(record["hidden_architecture"]),
            "model_type": record["model_type"],
            "repeat_idx": record["repeat_idx"],
            "orthogonal_init_seed": record["orthogonal_init_seed"],
            "dataloader_seed": record["dataloader_seed"],
            "epochs": record["epochs"],
            "final_train_accuracy": record["final_train_accuracy"],
            "final_test_accuracy": record["final_test_accuracy"],
            "best_test_accuracy": record["best_test_accuracy"],
            "final_train_loss": record["final_train_loss"],
            "final_test_loss": record["final_test_loss"],
            "elapsed_seconds": record["elapsed_seconds"],
            "result_path": record["result_path"],
        })

    return pd.DataFrame(rows)


def model_label(model_type):
    if model_type == "standard_tanh":
        return "standard-tanh"
    if model_type == "isotropic_tanh":
        return "isotropic-tanh"
    return model_type


def format_mean_se(values, include_n=False):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    n = int(values.size)

    if n == 0:
        return "--"

    mean = float(np.mean(values))

    if n >= 2:
        se = float(np.std(values, ddof=1) / math.sqrt(n))
        body = f"${mean:.2f} \\pm {se:.2f}$"
    else:
        body = f"${mean:.2f} \\pm \\text{{--}}$"

    if include_n:
        body = body[:-1] + f"\\;(n={n})$"

    return body


def build_accuracy_latex_table(records, value_key="final_test_accuracy", include_n=False, caption=None, label=None):
    table_records = [normalise_cached_record(record) for record in records]
    dataset_columns = DATASET_ORDER
    colspec = "l" + "cc" * len(dataset_columns)

    lines = []
    lines.append("\\begin{table}[ht]")
    lines.append("\\centering")

    if caption is not None:
        lines.append(f"\\caption{{{caption}}}")

    if label is not None:
        lines.append(f"\\label{{{label}}}")

    lines.append("\\begin{tabular}{" + colspec + "}")
    lines.append("\\toprule")

    header_one = ["Architecture"] + [
        f"\\multicolumn{{2}}{{c}}{{{dataset_name}}}"
        for dataset_name in dataset_columns
    ]
    lines.append(" & ".join(header_one) + " \\\\")

    header_two = [""] + [
        item
        for _ in dataset_columns
        for item in ["standard-tanh", "isotropic-tanh"]
    ]
    lines.append(" & ".join(header_two) + " \\\\")
    lines.append("\\midrule")

    for architecture_idx, hidden_architecture in enumerate(HIDDEN_ARCHITECTURES):
        architecture_label = architecture_label_from_hidden(hidden_architecture)
        hidden_tuple = architecture_tuple(hidden_architecture)

        row = [architecture_label]

        for dataset_name in dataset_columns:
            for model_type in MODEL_TYPES:
                values = [
                    record[value_key]
                    for record in table_records
                    if record["dataset_name"] == dataset_name
                    and architecture_tuple(record["hidden_architecture"]) == hidden_tuple
                    and record["model_type"] == model_type
                ]
                row.append(format_mean_se(values, include_n=include_n))

        lines.append(" & ".join(row) + " \\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("\\end{table}")

    return "\n".join(lines)


def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def save_summary_outputs(records, progress=False):
    records = [normalise_cached_record(record) for record in records]
    df = records_to_dataframe(records)

    if len(df) > 0:
        df.to_csv(SUMMARY_DIR / "Experiment6_CT256_raw_run_records.csv", index=False)

        summary = (
            df.groupby(["dataset_name", "architecture_idx", "architecture_label", "model_type"])
            .agg(
                n=("final_test_accuracy", "count"),
                mean_final_test_accuracy=("final_test_accuracy", "mean"),
                std_final_test_accuracy=("final_test_accuracy", "std"),
                mean_best_test_accuracy=("best_test_accuracy", "mean"),
                std_best_test_accuracy=("best_test_accuracy", "std"),
                mean_final_train_accuracy=("final_train_accuracy", "mean"),
                std_final_train_accuracy=("final_train_accuracy", "std"),
            )
            .reset_index()
        )

        summary["se_final_test_accuracy"] = summary["std_final_test_accuracy"] / np.sqrt(summary["n"])
        summary["se_best_test_accuracy"] = summary["std_best_test_accuracy"] / np.sqrt(summary["n"])
        summary.to_csv(SUMMARY_DIR / "Experiment6_CT256_summary_by_dataset_architecture_model.csv", index=False)

        paired = []
        key_cols = ["dataset_name", "architecture_idx", "repeat_idx"]

        for keys, group in df.groupby(key_cols):
            if set(group["model_type"]) >= {"standard_tanh", "isotropic_tanh"}:
                standard = group[group["model_type"] == "standard_tanh"].iloc[0]
                isotropic = group[group["model_type"] == "isotropic_tanh"].iloc[0]

                paired.append({
                    "dataset_name": keys[0],
                    "architecture_idx": keys[1],
                    "repeat_idx": keys[2],
                    "architecture_label": standard["architecture_label"],
                    "standard_final_test_accuracy": standard["final_test_accuracy"],
                    "isotropic_final_test_accuracy": isotropic["final_test_accuracy"],
                    "delta_final_test_accuracy": isotropic["final_test_accuracy"] - standard["final_test_accuracy"],
                    "standard_best_test_accuracy": standard["best_test_accuracy"],
                    "isotropic_best_test_accuracy": isotropic["best_test_accuracy"],
                    "delta_best_test_accuracy": isotropic["best_test_accuracy"] - standard["best_test_accuracy"],
                })

        if paired:
            pd.DataFrame(paired).to_csv(
                SUMMARY_DIR / "Experiment6_CT256_paired_isotropic_minus_standard.csv",
                index=False,
            )

    progress_tex = build_accuracy_latex_table(
        records,
        value_key="final_test_accuracy",
        include_n=True,
        caption="Experiment 6 progress table: final test accuracy mean $\\pm$ standard error where available.",
        label="tab:Experiment6_progress",
    )
    write_text(TABLES_DIR / "Experiment6_CT256_progress_accuracy_table.tex", progress_tex)
    write_text(TABLES_DIR / "_TEMPORARY_TABLE_.tex", progress_tex)

    final_tex = build_accuracy_latex_table(
        records,
        value_key="final_test_accuracy",
        include_n=False,
        caption="Experiment 6 final test accuracy: mean $\\pm$ standard error across repeats.",
        label="tab:Experiment6_final_accuracy",
    )
    write_text(TABLES_DIR / "Experiment6_CT256_final_accuracy_table.tex", final_tex)

    return progress_tex, final_tex


def display_progress(records, current_message=""):
    progress_tex, _ = save_summary_outputs(records, progress=True)

    clear_output(wait=True)

    completed = len(records)
    total = REPEATS * len(DATASET_ORDER) * len(HIDDEN_ARCHITECTURES) * len(MODEL_TYPES)
    message = f"Completed or loaded {completed}/{total} network runs."

    if current_message:
        message += f"\n\n{current_message}"

    display(Markdown("### Experiment 6 progress"))
    display(Markdown(message))
    display(Markdown("```tex\n" + progress_tex + "\n```"))


def train_or_load_run(dataset_name, architecture_idx, hidden_architecture, model_type, repeat_idx):
    hidden_architecture = list(hidden_architecture)
    architecture_label = architecture_label_from_hidden(hidden_architecture)

    result_path = run_result_path(
        dataset_name=dataset_name,
        architecture_idx=architecture_idx,
        hidden_architecture=hidden_architecture,
        model_type=model_type,
        repeat_idx=repeat_idx,
    )

    # First try the new architecture-labelled canonical filename.
    if result_path.exists():
        with open(result_path, "rb") as handle:
            record = pkl.load(handle)

        if record_matches_requested(
            record,
            dataset_name=dataset_name,
            hidden_architecture=hidden_architecture,
            model_type=model_type,
            repeat_idx=repeat_idx,
        ):
            record = normalise_cached_record(record)
            return record, True

        print(f"Cache mismatch detected, ignoring stale file: {result_path}")

    # Then search all cache files by their internal metadata.
    # This is what safely recovers older cache files whose filenames used stale positional arch indices.
    cached_record = find_matching_cached_record(
        dataset_name=dataset_name,
        hidden_architecture=hidden_architecture,
        model_type=model_type,
        repeat_idx=repeat_idx,
    )

    if cached_record is not None:
        cached_record = normalise_cached_record(cached_record)

        # Write a canonical copy using the new robust filename so future loads are direct.
        if not result_path.exists():
            with open(result_path, "wb") as handle:
                pkl.dump(cached_record, handle)

        return cached_record, True

    layers = full_architecture(dataset_name, hidden_architecture)
    orthogonal_init_seed = int(REPEAT_ORTHOGONAL_INIT_SEEDS[repeat_idx])
    dataloader_seed = int(REPEAT_DATALOADER_SEEDS[repeat_idx])
    train_loader, test_loader = make_dataloaders(dataset_name, repeat_idx)

    network = build_model(
        model_type=model_type,
        layers=layers,
        affine_seed=orthogonal_init_seed,
        device=DEVICE,
    )

    optimiser = torch.optim.AdamW(
        network.parameters(),
        lr=LEARNING_RATE,
        weight_decay=ADAMW_WEIGHT_DECAY,
    )

    test_x = [0]
    train_loss_history = []
    train_accuracy_history = []
    test_loss_history = []
    test_accuracy_history = []

    initial_test_loss, initial_test_accuracy = evaluate_classifier(network, test_loader, DEVICE, LOSS)
    test_loss_history.append(initial_test_loss)
    test_accuracy_history.append(initial_test_accuracy)

    start_time = time.time()

    for epoch in range(TOTAL_EPOCHS):
        train_loss, train_accuracy = train_one_epoch(network, train_loader, DEVICE, optimiser, LOSS)
        test_loss, test_accuracy = evaluate_classifier(network, test_loader, DEVICE, LOSS)

        train_loss_history.append(train_loss)
        train_accuracy_history.append(train_accuracy)
        test_x.append(epoch + 1)
        test_loss_history.append(test_loss)
        test_accuracy_history.append(test_accuracy)

        print(
            f"repeat={repeat_idx + 1}/{REPEATS}, dataset={dataset_name}, "
            f"arch={architecture_idx + 1}/{len(HIDDEN_ARCHITECTURES)} "
            f"({architecture_label}), model={model_label(model_type)}, "
            f"epoch={epoch + 1}/{TOTAL_EPOCHS}, test_acc={test_accuracy:.3f}%"
        )

    elapsed_seconds = time.time() - start_time

    state_path = str(
        model_state_path(
            dataset_name=dataset_name,
            architecture_idx=architecture_idx,
            hidden_architecture=hidden_architecture,
            model_type=model_type,
            repeat_idx=repeat_idx,
        )
    )

    if SAVE_MODEL_STATES:
        torch.save(network.cpu().state_dict(), state_path)
    else:
        state_path = None

    record = {
        "dataset_name": dataset_name,
        "architecture_idx": architecture_idx,
        "architecture_label": architecture_label,
        "architecture": layers,
        "hidden_architecture": list(hidden_architecture),
        "model_type": model_type,
        "repeat_idx": repeat_idx,
        "orthogonal_init_seed": orthogonal_init_seed,
        "dataloader_seed": dataloader_seed,
        "epochs": TOTAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "adamw_weight_decay": ADAMW_WEIGHT_DECAY,
        "weight_init": WEIGHT_INIT,
        "isotropic_intrinsic_length_approach": ISOTROPIC_INTRINSIC_LENGTH_APPROACH,
        "isotropic_linear_correction_approach": ISOTROPIC_LINEAR_CORRECTION_APPROACH,
        "test_x": np.asarray(test_x, dtype=np.float64),
        "train_loss_history": np.asarray(train_loss_history, dtype=np.float64),
        "train_accuracy_history": np.asarray(train_accuracy_history, dtype=np.float64),
        "test_loss_history": np.asarray(test_loss_history, dtype=np.float64),
        "test_accuracy_history": np.asarray(test_accuracy_history, dtype=np.float64),
        "final_train_accuracy": float(train_accuracy_history[-1]),
        "final_test_accuracy": float(test_accuracy_history[-1]),
        "best_test_accuracy": float(np.max(test_accuracy_history)),
        "final_train_loss": float(train_loss_history[-1]),
        "final_test_loss": float(test_loss_history[-1]),
        "elapsed_seconds": float(elapsed_seconds),
        "result_path": str(result_path),
        "model_state_path": state_path,
    }

    with open(result_path, "wb") as handle:
        pkl.dump(record, handle)

    return record, False

In [ ]:
# -------------------------
# Main experiment loop
# -------------------------

records_by_key = {}

for existing_record in load_existing_records():
    existing_record = normalise_cached_record(existing_record)

    key = cache_key(
        existing_record["dataset_name"],
        existing_record["hidden_architecture"],
        existing_record["model_type"],
        existing_record["repeat_idx"],
    )

    records_by_key[key] = existing_record

records = list(records_by_key.values())
display_progress(records, current_message="Loaded existing cached results before starting new work.")

for repeat_idx in range(REPEATS):
    for dataset_name in DATASET_ORDER:
        for architecture_idx, hidden_architecture in enumerate(HIDDEN_ARCHITECTURES):
            for model_type in MODEL_TYPES:
                record, was_cached = train_or_load_run(
                    dataset_name=dataset_name,
                    architecture_idx=architecture_idx,
                    hidden_architecture=hidden_architecture,
                    model_type=model_type,
                    repeat_idx=repeat_idx,
                )

                record = normalise_cached_record(record)

                key = cache_key(
                    dataset_name,
                    hidden_architecture,
                    model_type,
                    repeat_idx,
                )

                records_by_key[key] = record
                records = list(records_by_key.values())

                status = "loaded from cache" if was_cached else "trained and saved"

                display_progress(
                    records,
                    current_message=(
                        f"Last run {status}: repeat {repeat_idx + 1}/{REPEATS}, "
                        f"{dataset_name}, architecture {architecture_idx + 1}/{len(HIDDEN_ARCHITECTURES)} "
                        f"({record['architecture_label']}), {model_label(model_type)}, "
                        f"final test accuracy {record['final_test_accuracy']:.3f}%."
                    ),
                )

progress_tex, final_tex = save_summary_outputs(records, progress=False)

print("Experiment loop complete.")
print(f"Raw records: {SUMMARY_DIR / 'Experiment6_CT256_raw_run_records.csv'}")
print(f"Summary: {SUMMARY_DIR / 'Experiment6_CT256_summary_by_dataset_architecture_model.csv'}")
print(f"Paired deltas: {SUMMARY_DIR / 'Experiment6_CT256_paired_isotropic_minus_standard.csv'}")
print(f"Progress table: {TABLES_DIR / 'Experiment6_CT256_progress_accuracy_table.tex'}")
print(f"Final table: {TABLES_DIR / 'Experiment6_CT256_final_accuracy_table.tex'}")

In [ ]:

records = load_existing_records()
raw_df = records_to_dataframe(records)
display(raw_df.sort_values(["repeat_idx", "dataset_name", "architecture_idx", "model_type"]).head(20))

if len(raw_df) > 0:
    summary_df = (
        raw_df.groupby(["dataset_name", "architecture_idx", "architecture_label", "model_type"])
        .agg(
            n=("final_test_accuracy", "count"),
            mean_final_test_accuracy=("final_test_accuracy", "mean"),
            std_final_test_accuracy=("final_test_accuracy", "std"),
            mean_best_test_accuracy=("best_test_accuracy", "mean"),
            std_best_test_accuracy=("best_test_accuracy", "std"),
        )
        .reset_index()
    )
    summary_df["se_final_test_accuracy"] = summary_df["std_final_test_accuracy"] / np.sqrt(summary_df["n"])
    display(summary_df.sort_values(["dataset_name", "architecture_idx", "model_type"]))


In [ ]:
# -------------------------
# Final LaTeX table
# -------------------------

records = load_existing_records()
_, final_tex = save_summary_outputs(records, progress=False)
print(final_tex)
print(f"\nSaved to {TABLES_DIR / 'Experiment6_CT256_final_accuracy_table.tex'}")
